# Financial Markets SQL Datamart

Inserire il testo per spiegare il tutto, maxi riassunto generale

In [9]:
import os
import pandas as pd
import psycopg2
from dotenv import load_dotenv

load_dotenv()


def get_connection():
    return psycopg2.connect(
        host=os.getenv("PGHOST"),
        database=os.getenv("PGDATABASE"),
        user=os.getenv("PGUSER"),
        password=os.getenv("PGPASSWORD"),
        sslmode=os.getenv("PGSSLMODE")
    )


def execute_query(query):
    conn = None
    cur = None

    try:
        conn = get_connection()
        cur = conn.cursor()

        cur.execute(query)
        conn.commit()

        print("✓ Sweet! Data updated")

    except Exception as e:
        if conn:
            conn.rollback()

        print("Errore:", e)

    finally:
        if cur:
            cur.close()

        if conn:
            conn.close()


In [2]:
# CREAZIONE DEGLI SCHEMAS DEL PROGGETTO

# si provede con la creazione degli scheme per:

True

#Creazione degli schemas del progetto

creazione degli schemas per:
* raw
* staging
* ref
* core
* mart
* audit

procedo con l'inserimento anche di mart, tuttavia al suo interno non saranno presenti delle tabelle, al fine di gestire al meglio gli spazi, questo schema sarà popolato in un secondo momento, ovvero quando le altre tabelle saranno funzionanti e popolate, in modo da valutare al meglio se procedere con un eventuale aumento della memoria disponibile del fornitore o non procedere con l'upgrade

In [5]:
query = """
CREATE SCHEMA IF NOT EXISTS raw;
CREATE SCHEMA IF NOT EXISTS staging;
CREATE SCHEMA IF NOT EXISTS ref;
CREATE SCHEMA IF NOT EXISTS core;
CREATE SCHEMA IF NOT EXISTS mart;
CREATE SCHEMA IF NOT EXISTS audit;
"""

execute_query(query)
print ("schemas create con sucesso")


schemas create con sucesso


Ora si procede con la creazione delle prime tabelle, sullo schema REF.


In [ ]:
query = """
CREATE TABLE IF NOT EXISTS ref.currencies (
    currency_code       CHAR(3)         PRIMARY KEY,
    currency_name       VARCHAR(50)     NOT NULL,
    currency_symbol     VARCHAR(8),
    decimal_places      SMALLINT        NOT NULL,
    is_active           BOOLEAN         NOT NULL,

    CONSTRAINT chk_currency_code_upper
        CHECK (currency_code = UPPER(currency_code)),

    CONSTRAINT chk_currency_decimal_places
        CHECK (decimal_places >= 0)
);
"""

execute_query(query)


In [23]:
# creazione della tabella Currencies
query = """
INSERT INTO ref.currencies (
    currency_code,
    currency_name,
    currency_symbol,
    decimal_places,
    is_active
)
VALUES
    ('EUR', 'Euro', '€', 2, TRUE),
    ('USD', 'US Dollar', '$', 2, TRUE),
    ('GBP', 'British Pound', '£', 2, TRUE),
    ('CHF', 'Swiss Franc', 'CHF', 2, TRUE)
ON CONFLICT (currency_code) DO NOTHING;
"""
execute_query(query)
print("Tabella popolata con successo")

Operazione completata con successo
Tabella popolata con successo


In [24]:
#Creazione della tabella Calendar
query = """
CREATE TABLE IF NOT EXISTS ref.calendar (
    date_key            INTEGER			PRIMARY KEY,
    full_date	        DATE			NOT NULL,
    year_number	        SMALLINT		NOT NULL,
    quarter_number      SMALLINT		NOT NULL,
    month_number	    SMALLINT		NOT NULL,
    month_name	        VARCHAR(15)		NOT NULL,
    week_number	        SMALLINT		NOT NULL,
    day_of_month	    SMALLINT		NOT NULL,
    day_of_week	        SMALLINT		NOT NULL,
    is_weekend	        BOOLEAN			NOT NULL,
    is_month_end	    BOOLEAN			NOT NULL,
    is_quarter_end	    BOOLEAN			NOT NULL,
    is_year_end	        BOOLEAN			NOT NULL
)
"""
execute_query(query)

Operazione completata con successo


In [9]:
#REF.countries
query = """
CREATE TABLE IF NOT EXISTS ref.countries (
    country_code        CHAR(2)	        PRIMARY KEY,
    country_name	    VARCHAR(100)	NOT NULL,
    region_name	        VARCHAR(50),
    is_active	        BOOLEAN			NOT NULL
)
"""
execute_query(query)

✓ Sweet! Data updated


In [10]:
#REF.exchanges
query = """
CREATE TABLE IF NOT EXISTS ref.exchanges (
    exchange_mic	        CHAR(4)	        PRIMARY KEY,
    exchange_name	        VARCHAR(120)	NOT NULL,
    country_code	        CHAR(2),
    timezone_name	        VARCHAR(64),
    default_currency_code	CHAR(3),
    is_active	            BOOLEAN			NOT NULL,

    CONSTRAINT fk_country FOREIGN KEY (country_code)
        REFERENCES ref.countries(country_code),
    CONSTRAINT fk_currency FOREIGN KEY (default_currency_code)
        REFERENCES ref.currencies(currency_code)
)
"""
execute_query(query)

✓ Sweet! Data updated


In [12]:
# CORE.data_sources
query = """
CREATE TABLE IF NOT EXISTS Core.data_sources (
    source_id	            BIGINT GENERATED ALWAYS AS IDENTITY	    PRIMARY KEY,
    source_code	            VARCHAR(30)			                    NOT NULL	UNIQUE,
    source_name	            VARCHAR(120)			                NOT NULL,
    source_category	        VARCHAR(30)			                    NOT NULL,
    base_url	            TEXT,
    access_method	        VARCHAR(20)			                    NOT NULL,
    authentication_type	    VARCHAR(20)			                    NOT NULL,
    license_notes	        TEXT,
    redistribution_allowed	BOOLEAN			                        NOT NULL,
    rate_limit_notes	    VARCHAR(200),
    is_official	            BOOLEAN			                        NOT NULL,
    is_active	            BOOLEAN			                        NOT NULL,
    last_verified_at	    TIMESTAMPTZ
)
"""
execute_query(query)

✓ Sweet! Data updated


In [17]:
#CORE.clients
query = """
CREATE TABLE IF NOT EXISTS Core.clients (
    client_id	        BIGINT GENERATED ALWAYS AS IDENTITY			PRIMARY KEY,
    client_code	        VARCHAR(30)			                        NOT NULL UNIQUE,
    client_name	        VARCHAR(150)			                    NOT NULL,
    client_type	        VARCHAR(30)			                        NOT NULL  CHECK (client_type IN ('RETAIL','PRIVATE','INSTITUTIONAL','CORPORATE', 'OTHER')),
    segment	            VARCHAR(30),
    country_code	    CHAR(2),
    risk_profile	    VARCHAR(30),
    onboarding_date	    DATE			                            NOT NULL,
    status	            VARCHAR(20)			                        NOT NULL    CHECK ( status IN ('ACTIVE','INACTIVE','CLOSED')),
    created_at	        TIMESTAMPTZ			                        NOT NULL    DEFAULT CURRENT_TIMESTAMP,
    updated_at	        TIMESTAMPTZ,

    CONSTRAINT fk_country FOREIGN KEY (country_code)
        REFERENCES ref.countries(country_code)
)
"""
execute_query(query)

✓ Sweet! Data updated


In [19]:
#CORE.issuer
# ISSUER
query = """
CREATE TABLE IF NOT EXISTS Core.issuers (
    issuer_id	    BIGINT GENERATED ALWAYS AS IDENTITY 	PRIMARY KEY,
    issuer_code	    VARCHAR(30)			NOT NULL	                UNIQUE,
    issuer_name	    VARCHAR(150)		NOT NULL,
    issuer_type	    VARCHAR(30)			NOT NULL,
    country_code	CHAR(2),
    sector	        VARCHAR(50),
    industry	    VARCHAR(80),
    lei_code	    CHAR(20)			UNIQUE,
    is_active	    BOOLEAN			    NOT NULL,

    CONSTRAINT fk_country FOREIGN KEY (country_code)
        REFERENCES ref.countries(country_code)

)
"""
execute_query(query)

✓ Sweet! Data updated


In [24]:
#RAW.INSTURMENT_PRICE
# INSTRUMENT_PRICE
query = """
CREATE TABLE raw.instrument_prices (
    -- Chiave tecnica
    raw_price_id           BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Identificativi strumento
    ticker                 VARCHAR(30),
    isin                   CHAR(12),

    -- Data del prezzo
    price_date             DATE NOT NULL,

    -- Prezzi
    open_price             NUMERIC(20,8),
    high_price             NUMERIC(20,8),
    low_price              NUMERIC(20,8),
    close_price            NUMERIC(20,8),
    adjusted_close         NUMERIC(20,8),

    -- Volume
    volume                 BIGINT,

    -- Valuta
    currency               CHAR(3),

    -- Lineage tecnico / metadati
    source_name            VARCHAR(100),
    source_file            VARCHAR(255),
    batch_id               BIGINT,
    loaded_at              TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);
CREATE INDEX IF NOT EXISTS idx_raw_instrument_prices_lookup
ON raw.instrument_prices (ticker, price_date);

CREATE INDEX IF NOT EXISTS idx_raw_instrument_prices_isin
ON raw.instrument_prices (isin);
"""
execute_query(query)

✓ Sweet! Data updated


In [6]:
#RAW.fx_rates
# FOREX RATES
query = """
CREATE TABLE raw.fx_rates (
    rate_date               DATE NOT NULL,
    base_currency           CHAR(3) NOT NULL,
    quote_currency          CHAR(3) NOT NULL,
    fx_rate                 NUMERIC(20, 10) NOT NULL,
    rate_type               VARCHAR(20) DEFAULT 'CLOSE',
    source_name             VARCHAR(50),
    ingestion_file_id       BIGINT,
    load_at                 TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT chk_raw_fx_rate CHECK (fx_rate > 0)
);

CREATE INDEX idx_raw_fx_rates_lookup
ON raw.fx_rates (rate_date, base_currency, quote_currency);
"""
execute_query(query)

✓ Sweet! Data updated


In [7]:
#RAW.TRANSACTIONS
# TRANSACTIONS
query = """
CREATE TABLE raw.transactions (
    -- Chiavi tecniche e identificativi
    raw_transaction_id     BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    source_transaction_id  VARCHAR(100),

    -- Dettagli Portafoglio e Cliente
    client_code            VARCHAR(50),
    account_code           VARCHAR(50),
    portfolio_code         VARCHAR(50),

    -- Date di Business
    transaction_date       DATE,
    settlement_date        DATE,
    trade_date             DATE,

    -- Anagrafica Strumento
    isin                   CHAR(12),
    ticker                 VARCHAR(30),
    source_instrument_id   VARCHAR(100),

    -- Dettagli Operazione
    transaction_type       VARCHAR(50),
    quantity               NUMERIC(24,8),
    price                  NUMERIC(20,8),

    -- Importi e Costi
    gross_amount           NUMERIC(24,8),
    net_amount             NUMERIC(24,8),
    commission             NUMERIC(20,8),
    fees                   NUMERIC(20,8),
    taxes                  NUMERIC(20,8),

    -- Valuta e FX
    currency               CHAR(3),
    fx_rate                NUMERIC(20,10),
    cash_amount            NUMERIC(24,8),

    -- Altri dettagli
    counterparty           VARCHAR(100),
    market                 VARCHAR(50),
    description            TEXT,

    -- Lineage tecnico / Meta-dati di caricamento
    source_name            VARCHAR(100),
    source_file            VARCHAR(255),
    batch_id               BIGINT,
    loaded_at              TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);
"""
execute_query(query)


✓ Sweet! Data updated


In [8]:
#RAW.orporate_actions
# orporate_actions
query = """
CREATE TABLE raw.corporate_actions (
    -- Chiavi tecniche e identificativi
    raw_corporate_action_id  BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    source_action_id         VARCHAR(100),

    -- Anagrafica Strumento
    isin                     CHAR(12),
    ticker                   VARCHAR(30),
    source_instrument_id     VARCHAR(100),

    -- Dettagli Evento
    action_type              VARCHAR(50),

    -- Date di Business
    announcement_date        DATE,
    ex_date                  DATE,
    record_date              DATE,
    payment_date             DATE,

    -- Importi e Rapporti
    currency                 CHAR(3),
    cash_amount              NUMERIC(20,8),
    dividend_per_share       NUMERIC(20,8),
    ratio_from               NUMERIC(18,8),
    ratio_to                 NUMERIC(18,8),

    -- Dati per cambi strumento (es. Merger / Ticker change)
    new_isin                 CHAR(12),
    new_ticker               VARCHAR(30),

    -- Altri dettagli
    description              TEXT,

    -- Lineage tecnico / Meta-dati di caricamento
    source_name              VARCHAR(100),
    source_file              VARCHAR(255),
    batch_id                 BIGINT,
    loaded_at                TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);
"""
execute_query(query)

✓ Sweet! Data updated


In [9]:
#RAW.benchmarks
# benchmarks
query = """
CREATE TABLE raw.benchmarks (
    -- Chiavi tecniche e identificativi
    raw_benchmark_id     BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    source_benchmark_id  VARCHAR(100),

    -- Dettagli Benchmark
    benchmark_code       VARCHAR(30),
    benchmark_name       VARCHAR(150),

    -- Identificativi di mercato
    ticker               VARCHAR(30),
    isin                 CHAR(12),

    -- Tipologia e classificazione
    benchmark_type       VARCHAR(30),
    currency             CHAR(3),
    country_code         CHAR(2),
    provider             VARCHAR(50),
    asset_class          VARCHAR(40),

    -- Altri dettagli descrittivi
    description          TEXT,

    -- Lineage tecnico / Meta-dati di caricamento
    source_name          VARCHAR(100),
    source_file          VARCHAR(255),
    batch_id             BIGINT,
    loaded_at            TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);
"""
execute_query(query)

✓ Sweet! Data updated


In [10]:
#RAW.benchmark_prices
# benchmark_prices
query = """
CREATE TABLE IF NOT EXISTS raw.benchmark_prices (
    -- Chiavi tecniche
    raw_benchmark_price_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Riferimenti Benchmark
    benchmark_code         VARCHAR(30),
    benchmark_ticker       VARCHAR(30),

    -- Data di osservazione
    price_date             DATE,

    -- Valori / Livelli Indice (OHLC)
    open_value             NUMERIC(20,8),
    high_value             NUMERIC(20,8),
    low_value              NUMERIC(20,8),
    close_value            NUMERIC(20,8),
    adjusted_close_value   NUMERIC(20,8),

    -- Valuta
    currency               CHAR(3),

    -- Lineage tecnico / Meta-dati di caricamento
    source_name            VARCHAR(100),
    source_file            VARCHAR(255),
    batch_id               BIGINT,
    loaded_at              TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);
"""

execute_query(query)

✓ Sweet! Data updated


In [11]:
#RAW.positions
# positions
query = """
CREATE TABLE IF NOT EXISTS raw.positions (
    -- Chiavi tecniche
    raw_position_id        BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Data di validità del saldo
    position_date          DATE,

    -- Riferimenti Portafoglio e Cliente
    client_code            VARCHAR(50),
    portfolio_code         VARCHAR(50),
    account_code           VARCHAR(50),

    -- Identificativi Strumento
    isin                   CHAR(12),
    ticker                 VARCHAR(30),
    source_instrument_id   VARCHAR(100),

    -- Quantità
    quantity               NUMERIC(24,8),

    -- Valorizzazione di Mercato
    market_price           NUMERIC(20,8),
    market_value           NUMERIC(24,8),

    -- Valorizzazione di Costo
    cost_price             NUMERIC(20,8),
    cost_value             NUMERIC(24,8),

    -- Profitti e Perdite (P&L)
    unrealized_pnl         NUMERIC(24,8),

    -- Valuta
    currency               CHAR(3),

    -- Lineage tecnico / Meta-dati di caricamento
    source_name            VARCHAR(100),
    source_file            VARCHAR(255),
    batch_id               BIGINT,
    loaded_at              TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
);
"""
execute_query(query)

✓ Sweet! Data updated


Inserimento tabelle sullo schema STAGING

In [12]:
query = """
CREATE TABLE IF NOT EXISTS staging.instrument_prices (
    -- Chiave tecnica STAGING
    staging_price_id      BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Lineage verso RAW
    raw_price_id          BIGINT NOT NULL UNIQUE,

    -- Identificativi strumento normalizzati
    ticker                VARCHAR(30),
    isin                  VARCHAR(12),

    -- Data convertita
    price_date            DATE NOT NULL,

    -- Prezzi convertiti
    open_price            NUMERIC(20,8),
    high_price            NUMERIC(20,8),
    low_price             NUMERIC(20,8),
    close_price           NUMERIC(20,8),
    adjusted_close        NUMERIC(20,8),

    -- Volume
    volume                BIGINT,

    -- Valuta
    currency              CHAR(3),

    -- Fonte
    source_name           VARCHAR(100),

    -- Lineage temporale
    raw_loaded_at         TIMESTAMP,
    staged_at             TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);
"""
execute_query(query)

✓ Sweet! Data updated


In [13]:
query = """
CREATE TABLE IF NOT EXISTS staging.instrument_prices (
    -- Chiave tecnica STAGING
    staging_price_id      BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Lineage verso RAW
    raw_price_id          BIGINT NOT NULL UNIQUE,

    -- Identificativi strumento normalizzati
    ticker                VARCHAR(30),
    isin                  VARCHAR(12),

    -- Data convertita
    price_date            DATE NOT NULL,

    -- Prezzi convertiti
    open_price            NUMERIC(20,8),
    high_price            NUMERIC(20,8),
    low_price             NUMERIC(20,8),
    close_price           NUMERIC(20,8),
    adjusted_close        NUMERIC(20,8),

    -- Volume
    volume                BIGINT,

    -- Valuta
    currency              CHAR(3),

    -- Fonte
    source_name           VARCHAR(100),

    -- Lineage temporale
    raw_loaded_at         TIMESTAMP,
    staged_at             TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);
"""
execute_query(query)

✓ Sweet! Data updated


In [14]:
query = """
CREATE TABLE IF NOT EXISTS staging.transactions (
    staging_transaction_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Lineage RAW
    raw_transaction_id       BIGINT NOT NULL UNIQUE,

    -- Identificativo sorgente
    source_transaction_id    VARCHAR(100),

    -- Cliente / Portafoglio
    client_code              VARCHAR(50),
    account_code             VARCHAR(50),
    portfolio_code           VARCHAR(50),

    -- Date
    transaction_date         DATE,
    settlement_date          DATE,
    trade_date               DATE,

    -- Strumento
    isin                     VARCHAR(12),
    ticker                   VARCHAR(30),
    source_instrument_id     VARCHAR(100),

    -- Operazione normalizzata
    transaction_type         VARCHAR(50),

    quantity                 NUMERIC(24,8),
    price                    NUMERIC(20,8),

    -- Importi
    gross_amount             NUMERIC(24,8),
    net_amount               NUMERIC(24,8),

    commission               NUMERIC(20,8),
    fees                     NUMERIC(20,8),
    taxes                    NUMERIC(20,8),

    -- Valuta
    currency                 CHAR(3),
    fx_rate                  NUMERIC(20,10),

    cash_amount              NUMERIC(24,8),

    -- Informazioni operative
    counterparty             VARCHAR(100),
    market                   VARCHAR(50),
    description              TEXT,

    -- Lineage
    source_name              VARCHAR(100),
    source_file              VARCHAR(255),
    batch_id                 BIGINT,
    raw_loaded_at            TIMESTAMPTZ,

    staged_at                TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_stg_transactions_isin
ON staging.transactions (isin);

CREATE INDEX IF NOT EXISTS idx_stg_transactions_portfolio
ON staging.transactions (portfolio_code);

CREATE INDEX IF NOT EXISTS idx_stg_transactions_trade_date
ON staging.transactions (trade_date);
"""
execute_query(query)

✓ Sweet! Data updated


In [15]:
query = """
CREATE TABLE IF NOT EXISTS staging.corporate_actions (
    staging_corporate_action_id  BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Lineage RAW
    raw_corporate_action_id      BIGINT NOT NULL UNIQUE,

    source_action_id             VARCHAR(100),

    -- Strumento
    isin                         VARCHAR(12),
    ticker                       VARCHAR(30),
    source_instrument_id         VARCHAR(100),

    -- Evento normalizzato
    action_type                  VARCHAR(50),

    -- Date
    announcement_date            DATE,
    ex_date                      DATE,
    record_date                  DATE,
    payment_date                 DATE,

    -- Importi
    currency                     CHAR(3),
    cash_amount                  NUMERIC(20,8),
    dividend_per_share           NUMERIC(20,8),

    ratio_from                   NUMERIC(18,8),
    ratio_to                     NUMERIC(18,8),

    -- Eventuale nuovo strumento
    new_isin                     VARCHAR(12),
    new_ticker                   VARCHAR(30),

    description                  TEXT,

    -- Lineage
    source_name                  VARCHAR(100),
    source_file                  VARCHAR(255),
    batch_id                     BIGINT,
    raw_loaded_at                TIMESTAMPTZ,

    staged_at                    TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_stg_corporate_actions_isin
ON staging.corporate_actions (isin);

CREATE INDEX IF NOT EXISTS idx_stg_corporate_actions_ex_date
ON staging.corporate_actions (ex_date);
"""
execute_query(query)

✓ Sweet! Data updated


In [16]:
query = """
CREATE TABLE IF NOT EXISTS staging.corporate_actions (
    staging_corporate_action_id  BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Lineage RAW
    raw_corporate_action_id      BIGINT NOT NULL UNIQUE,

    source_action_id             VARCHAR(100),

    -- Strumento
    isin                         VARCHAR(12),
    ticker                       VARCHAR(30),
    source_instrument_id         VARCHAR(100),

    -- Evento normalizzato
    action_type                  VARCHAR(50),

    -- Date
    announcement_date            DATE,
    ex_date                      DATE,
    record_date                  DATE,
    payment_date                 DATE,

    -- Importi
    currency                     CHAR(3),
    cash_amount                  NUMERIC(20,8),
    dividend_per_share           NUMERIC(20,8),

    ratio_from                   NUMERIC(18,8),
    ratio_to                     NUMERIC(18,8),

    -- Eventuale nuovo strumento
    new_isin                     VARCHAR(12),
    new_ticker                   VARCHAR(30),

    description                  TEXT,

    -- Lineage
    source_name                  VARCHAR(100),
    source_file                  VARCHAR(255),
    batch_id                     BIGINT,
    raw_loaded_at                TIMESTAMPTZ,

    staged_at                    TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_stg_corporate_actions_isin
ON staging.corporate_actions (isin);

CREATE INDEX IF NOT EXISTS idx_stg_corporate_actions_ex_date
ON staging.corporate_actions (ex_date);
"""
execute_query(query)

✓ Sweet! Data updated


In [17]:
query = """
CREATE TABLE IF NOT EXISTS staging.benchmarks (
    staging_benchmark_id    BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Lineage RAW
    raw_benchmark_id        BIGINT NOT NULL UNIQUE,

    source_benchmark_id     VARCHAR(100),

    -- Benchmark
    benchmark_code          VARCHAR(30) NOT NULL,
    benchmark_name          VARCHAR(150) NOT NULL,

    -- Identificativi
    ticker                  VARCHAR(30),
    isin                    VARCHAR(12),

    -- Classificazione
    benchmark_type          VARCHAR(30),
    currency                CHAR(3),
    country_code            CHAR(2),

    provider                VARCHAR(50),
    asset_class             VARCHAR(40),

    description             TEXT,

    -- Lineage
    source_name             VARCHAR(100),
    source_file             VARCHAR(255),
    batch_id                BIGINT,
    raw_loaded_at           TIMESTAMPTZ,

    staged_at               TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_stg_benchmarks_code
ON staging.benchmarks (benchmark_code);
"""
execute_query(query)

✓ Sweet! Data updated


In [18]:
query = """
CREATE TABLE IF NOT EXISTS staging.benchmark_prices (
    staging_benchmark_price_id  BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Lineage RAW
    raw_benchmark_price_id      BIGINT NOT NULL UNIQUE,

    -- Benchmark
    benchmark_code              VARCHAR(30),
    benchmark_ticker            VARCHAR(30),

    -- Data
    price_date                  DATE NOT NULL,

    -- Valori
    open_value                  NUMERIC(20,8),
    high_value                  NUMERIC(20,8),
    low_value                   NUMERIC(20,8),
    close_value                 NUMERIC(20,8),
    adjusted_close_value        NUMERIC(20,8),

    currency                    CHAR(3),

    -- Lineage
    source_name                 VARCHAR(100),
    source_file                 VARCHAR(255),
    batch_id                    BIGINT,
    raw_loaded_at               TIMESTAMPTZ,

    staged_at                   TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_stg_benchmark_prices_lookup
ON staging.benchmark_prices (
    benchmark_code,
    price_date
);
"""
execute_query(query)

✓ Sweet! Data updated


In [19]:
query = """
CREATE TABLE IF NOT EXISTS staging.positions (
    staging_position_id      BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Lineage RAW
    raw_position_id          BIGINT NOT NULL UNIQUE,

    -- Data posizione
    position_date            DATE NOT NULL,

    -- Cliente / portafoglio
    client_code              VARCHAR(50),
    portfolio_code           VARCHAR(50),
    account_code             VARCHAR(50),

    -- Strumento
    isin                     VARCHAR(12),
    ticker                   VARCHAR(30),
    source_instrument_id     VARCHAR(100),

    -- Quantità
    quantity                 NUMERIC(24,8),

    -- Valore di mercato
    market_price             NUMERIC(20,8),
    market_value             NUMERIC(24,8),

    -- Costo
    cost_price               NUMERIC(20,8),
    cost_value               NUMERIC(24,8),

    -- P&L
    unrealized_pnl           NUMERIC(24,8),

    currency                 CHAR(3),

    -- Lineage
    source_name              VARCHAR(100),
    source_file              VARCHAR(255),
    batch_id                 BIGINT,
    raw_loaded_at            TIMESTAMPTZ,

    staged_at                TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_stg_positions_portfolio_date
ON staging.positions (
    portfolio_code,
    position_date
);

CREATE INDEX IF NOT EXISTS idx_stg_positions_isin
ON staging.positions (isin);
"""
execute_query(query)

✓ Sweet! Data updated


In [20]:
query = """
CREATE TABLE IF NOT EXISTS staging.fx_rates (
    staging_fx_rate_id      BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    -- Lineage RAW
    raw_fx_rate_id          BIGINT NOT NULL UNIQUE,

    -- Business data
    rate_date               DATE NOT NULL,
    base_currency           CHAR(3) NOT NULL,
    quote_currency          CHAR(3) NOT NULL,

    fx_rate                 NUMERIC(20,10) NOT NULL,

    rate_type               VARCHAR(20) NOT NULL DEFAULT 'CLOSE',

    -- Fonte
    source_name             VARCHAR(50),
    ingestion_file_id       BIGINT,

    -- Lineage temporale
    raw_loaded_at           TIMESTAMPTZ,
    staged_at               TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP,

    CONSTRAINT chk_stg_fx_rate_positive
        CHECK (fx_rate > 0),

    CONSTRAINT chk_stg_fx_different_currencies
        CHECK (base_currency <> quote_currency)
);

CREATE INDEX IF NOT EXISTS idx_stg_fx_rates_lookup
ON staging.fx_rates (
    rate_date,
    base_currency,
    quote_currency
);
"""
execute_query(query)

✓ Sweet! Data updated


In [4]:
query =(
"""
CREATE TABLE IF NOT EXISTS audit.etl_batches (
    batch_id                BIGINT         GENERATED ALWAYS AS IDENTITY NOT NULL,
    process_name            VARCHAR(100)       NOT NULL,
    source_id BIGINT,
    started_at              TIMESTAMPTZ          NOT NULL DEFAULT CURRENT_TIMESTAMP,
    finished_at             TIMESTAMPTZ,
    status                  VARCHAR(20)      NOT NULL DEFAULT 'RUNNING',
    rows_read               BIGINT,
    rows_inserted           BIGINT,
    rows_updated            BIGINT,
    rows_rejected           BIGINT,
    error_message           TEXT,
    CONSTRAINT pk_etl_batches PRIMARY KEY (batch_id),
    CONSTRAINT fk_etl_batches_source_id FOREIGN KEY (source_id) REFERENCES core.data_sources(source_id),
    CONSTRAINT ck_etl_batches_finished_at CHECK (finished_at IS NULL OR finished_at >= started_at),
    CONSTRAINT ck_etl_batches_status CHECK (status IN ('RUNNING','SUCCESS','WARNING','FAILED')),
    CONSTRAINT ck_etl_batches_rows_read CHECK (rows_read IS NULL OR rows_read >= 0),
    CONSTRAINT ck_etl_batches_rows_inserted CHECK (rows_inserted IS NULL OR rows_inserted >= 0),
    CONSTRAINT ck_etl_batches_rows_updated CHECK (rows_updated IS NULL OR rows_updated >= 0),
    CONSTRAINT ck_etl_batches_rows_rejected CHECK (rows_rejected IS NULL OR rows_rejected >= 0)
);

CREATE INDEX IF NOT EXISTS ix_etl_batches_status_started_at
    ON audit.etl_batches (status, started_at);
"""
)

execute_query(query)

✓ Sweet! Data updated


In [5]:

query = (
"""
CREATE TABLE IF NOT EXISTS audit.ingestion_files (
    ingestion_file_id           BIGINT GENERATED ALWAYS AS IDENTITY NOT NULL,
    batch_id                    BIGINT                              NOT NULL,
    source_id                   BIGINT                              NOT NULL,
    file_name                   VARCHAR(255)                        NOT NULL,
    raw_path TEXT               NOT NULL,
    content_checksum            CHAR(64)                            NOT NULL,
    extracted_at                TIMESTAMPTZ                         NOT NULL DEFAULT CURRENT_TIMESTAMP,
    period_start                DATE,
    period_end                  DATE,
    row_count                   BIGINT,
    load_status                 VARCHAR(20)                         NOT NULL,
    error_message               TEXT,

    CONSTRAINT pk_ingestion_files PRIMARY KEY (ingestion_file_id),
    CONSTRAINT fk_ingestion_files_batch_id FOREIGN KEY (batch_id) REFERENCES audit.etl_batches(batch_id),
    CONSTRAINT fk_ingestion_files_source_id FOREIGN KEY (source_id) REFERENCES core.data_sources(source_id),
    CONSTRAINT uq_ingestion_files_content_checksum UNIQUE (content_checksum),
    CONSTRAINT ck_ingestion_files_content_checksum CHECK (length(content_checksum)=64),
    CONSTRAINT ck_ingestion_files_period_end CHECK (period_end IS NULL OR period_start IS NULL OR period_end >= period_start),
    CONSTRAINT ck_ingestion_files_row_count CHECK (row_count IS NULL OR row_count >= 0),
    CONSTRAINT ck_ingestion_files_load_status CHECK (load_status IN ('DOWNLOADED','VALIDATED','LOADED','REJECTED'))
);

CREATE INDEX IF NOT EXISTS ix_ingestion_files_source_extracted_at
    ON audit.ingestion_files (source_id, extracted_at);

CREATE INDEX IF NOT EXISTS ix_ingestion_files_batch_id
    ON audit.ingestion_files (batch_id);
"""
)

execute_query(query)

✓ Sweet! Data updated


In [12]:
query = (
"""
CREATE TABLE IF NOT EXISTS audit.data_quality_rules (
    rule_code       VARCHAR(50)     NOT NULL,
    rule_name       VARCHAR(150)    NOT NULL,
    target_table    VARCHAR(120)    NOT NULL,
    description     TEXT            NOT NULL,
    severity        VARCHAR(20)     NOT NULL,
    blocking_flag   BOOLEAN         NOT NULL DEFAULT FALSE,

    tolerance_value NUMERIC(24,10),
    tolerance_unit  VARCHAR(30),
    is_enabled      BOOLEAN         NOT NULL DEFAULT TRUE,

    created_at TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMPTZ,

    CONSTRAINT pk_data_quality_rules PRIMARY KEY (rule_code),
    CONSTRAINT ck_data_quality_rules_severity CHECK (severity IN ('INFO','WARNING','ERROR','CRITICAL')),
    CONSTRAINT ck_data_quality_rules_tolerance_value CHECK (tolerance_value IS NULL OR tolerance_value >= 0),
    CONSTRAINT ck_dq_rules_tolerance_unit CHECK (tolerance_value IS NULL OR tolerance_unit IS NOT NULL)
);

"""
)
execute_query(query)

✓ Sweet! Data updated


In [3]:

query = (
"""
CREATE TABLE IF NOT EXISTS audit.data_quality_issues (
    dq_issue_id         BIGINT GENERATED ALWAYS AS IDENTITY     NOT NULL,
    batch_id            BIGINT,
    rule_code           VARCHAR(50)                             NOT NULL,
    severity            VARCHAR(20)                             NOT NULL,
    table_name          VARCHAR(120)                            NOT NULL,
    record_key          VARCHAR(250),
    issue_date          DATE,
    issue_description   VARCHAR(500)                            NOT NULL,
    detected_at         TIMESTAMPTZ                             NOT NULL DEFAULT CURRENT_TIMESTAMP,
    status              VARCHAR(20)                             NOT NULL DEFAULT 'OPEN',
    resolved_at         TIMESTAMPTZ,
    resolution_note     VARCHAR(500),

    CONSTRAINT pk_data_quality_issues PRIMARY KEY (dq_issue_id),
    CONSTRAINT fk_data_quality_issues_batch_id FOREIGN KEY (batch_id) REFERENCES audit.etl_batches(batch_id),
    CONSTRAINT fk_data_quality_issues_rule_code FOREIGN KEY (rule_code) REFERENCES audit.data_quality_rules(rule_code),
    CONSTRAINT ck_data_quality_issues_severity CHECK (severity IN ('INFO','WARNING','ERROR','CRITICAL')),
    CONSTRAINT ck_data_quality_issues_status CHECK (status IN ('OPEN','ACCEPTED','RESOLVED')),
    CONSTRAINT ck_dq_issues_resolved_at CHECK (status <> 'RESOLVED' OR resolved_at IS NOT NULL)
);

CREATE INDEX IF NOT EXISTS ix_dq_issues_rule_status
    ON audit.data_quality_issues (rule_code, status);

CREATE INDEX IF NOT EXISTS ix_dq_issues_severity_detected_at
    ON audit.data_quality_issues (severity, detected_at);

CREATE INDEX IF NOT EXISTS ix_dq_issues_batch_id
    ON audit.data_quality_issues (batch_id);

"""
)

execute_query(query)


✓ Sweet! Data updated


In [12]:
query ="""
CREATE TABLE IF NOT EXISTS core.portfolios (
    portfolio_id                  BIGINT GENERATED ALWAYS AS IDENTITY         PRIMARY KEY,
    portfolio_code                VARCHAR(30)                                 NOT NULL UNIQUE,
    client_id                     BIGINT                                      NOT NULL,
    portfolio_name                VARCHAR(100)                                NOT NULL,
    portfolio_type                VARCHAR(30)                                 NOT NULL,
    investment_strategy           VARCHAR(50),
    management_type               VARCHAR(30),
    base_currency_code            CHAR(3)                                     NOT NULL,
    inception_date                DATE                                        NOT NULL,
    closure_date                  DATE,
    risk_level                    SMALLINT,
    status                        VARCHAR(20)                                 NOT NULL DEFAULT 'ACTIVE',
    created_at                    TIMESTAMPTZ                                 NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at                    TIMESTAMPTZ,

    CONSTRAINT fk_portfolios_client_id
        FOREIGN KEY (client_id)
        REFERENCES core.clients(client_id),

    CONSTRAINT chk_portfolios_portfolio_type
        CHECK (portfolio_type IN ('ADVISORY','MANAGED','SELF_DIRECTED','FUND')),

    CONSTRAINT chk_portfolios_management_type
        CHECK (management_type IS NULL OR management_type IN ('ACTIVE','PASSIVE','MIXED')),

    CONSTRAINT fk_portfolios_base_currency_code
        FOREIGN KEY (base_currency_code)
        REFERENCES ref.currencies(currency_code),

    CONSTRAINT chk_portfolios_closure_date
        CHECK (closure_date IS NULL OR closure_date >= inception_date),

    CONSTRAINT chk_portfolios_risk_level
        CHECK (risk_level IS NULL OR risk_level BETWEEN 1 AND 7),

    CONSTRAINT chk_portfolios_status
        CHECK (status IN ('ACTIVE','SUSPENDED','CLOSED'))
);

CREATE INDEX IF NOT EXISTS idx_portfolios_1
ON core.portfolios (client_id, status);
"""

execute_query(query)



✓ Sweet! Data updated


In [13]:
# CORE.benchmarks

query = """
CREATE TABLE IF NOT EXISTS core.benchmarks (
    benchmark_id                  BIGINT GENERATED ALWAYS AS IDENTITY         PRIMARY KEY,
    benchmark_code                VARCHAR(30)                                 NOT NULL UNIQUE,
    benchmark_name                VARCHAR(150)                                NOT NULL,
    provider_name                 VARCHAR(50),
    benchmark_type                VARCHAR(30)                                 NOT NULL,
    return_type                   VARCHAR(20)                                 NOT NULL,
    currency_code                 CHAR(3)                                     NOT NULL,
    geographic_area               VARCHAR(40),
    asset_class                   VARCHAR(40),
    base_date                     DATE,
    is_composite                  BOOLEAN                                     NOT NULL DEFAULT FALSE,
    is_active                     BOOLEAN                                     NOT NULL DEFAULT TRUE,

    CONSTRAINT chk_benchmarks_benchmark_type
        CHECK (benchmark_type IN ('EQUITY','BOND','BALANCED','MONEY_MARKET','CUSTOM')),

    CONSTRAINT chk_benchmarks_return_type
        CHECK (return_type IN ('PRICE','GROSS_TR','NET_TR')),

    CONSTRAINT fk_benchmarks_currency_code
        FOREIGN KEY (currency_code)
        REFERENCES ref.currencies(currency_code)
);
"""
execute_query(query)

✓ Sweet! Data updated


In [14]:
# CORE.instruments

query = """
CREATE TABLE IF NOT EXISTS core.instruments (
    instrument_id                 BIGINT GENERATED ALWAYS AS IDENTITY         PRIMARY KEY,
    instrument_code               VARCHAR(40)                                 NOT NULL UNIQUE,
    instrument_name               VARCHAR(200)                                NOT NULL,
    short_name                    VARCHAR(80),
    instrument_type               VARCHAR(30)                                 NOT NULL,
    instrument_subtype            VARCHAR(40)                                 NOT NULL,
    isin                          CHAR(12),
    ticker                        VARCHAR(30),
    issuer_id                     BIGINT,
    trading_currency_code         CHAR(3)                                     NOT NULL,
    country_of_issue              CHAR(2),
    primary_exchange_mic          CHAR(4),
    issue_date                    DATE,
    maturity_date                 DATE,
    status                        VARCHAR(20)                                 NOT NULL DEFAULT 'ACTIVE',
    price_multiplier              NUMERIC(18,8)                               NOT NULL DEFAULT 1.0,
    created_at                    TIMESTAMPTZ                                 NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at                    TIMESTAMPTZ,

    CONSTRAINT chk_instruments_instrument_type
        CHECK (instrument_type IN ('EQUITY','ETF','BOND','FUND','CASH_EQUIVALENT')),

    CONSTRAINT chk_instruments_isin
        CHECK (isin IS NULL OR length(isin)=12),

    CONSTRAINT fk_instruments_issuer_id
        FOREIGN KEY (issuer_id)
        REFERENCES core.issuers(issuer_id),

    CONSTRAINT fk_instruments_trading_currency_code
        FOREIGN KEY (trading_currency_code)
        REFERENCES ref.currencies(currency_code),

    CONSTRAINT fk_instruments_country_of_issue
        FOREIGN KEY (country_of_issue)
        REFERENCES ref.countries(country_code),

    CONSTRAINT fk_instruments_primary_exchange_mic
        FOREIGN KEY (primary_exchange_mic)
        REFERENCES ref.exchanges(exchange_mic),

    CONSTRAINT chk_instruments_maturity_date
        CHECK (maturity_date IS NULL OR issue_date IS NULL OR maturity_date >= issue_date),

    CONSTRAINT chk_instruments_status
        CHECK (status IN ('ACTIVE','MATURED','DELISTED','INACTIVE')),

    CONSTRAINT chk_instruments_price_multiplier
        CHECK (price_multiplier > 0)
);

CREATE UNIQUE INDEX IF NOT EXISTS uq_instruments_3
ON core.instruments (isin)
WHERE isin IS NOT NULL;

CREATE INDEX IF NOT EXISTS idx_instruments_1
ON core.instruments (instrument_type, instrument_subtype);

CREATE INDEX IF NOT EXISTS idx_instruments_2
ON core.instruments (issuer_id);

CREATE INDEX IF NOT EXISTS idx_instruments_3
ON core.instruments (ticker, primary_exchange_mic);
"""

execute_query(query)

✓ Sweet! Data updated


In [15]:
# CORE.instrument_identifiers

query = """
CREATE TABLE IF NOT EXISTS core.instrument_identifiers (
    instrument_identifier_id      BIGINT GENERATED ALWAYS AS IDENTITY         PRIMARY KEY,
    instrument_id                 BIGINT                                      NOT NULL,
    identifier_type               VARCHAR(20)                                 NOT NULL,
    identifier_value              VARCHAR(80)                                 NOT NULL,
    exchange_mic                  CHAR(4),
    source_id                     BIGINT,
    valid_from                    DATE                                        NOT NULL,
    valid_to                      DATE,
    is_primary                    BOOLEAN                                     NOT NULL DEFAULT FALSE,

    CONSTRAINT uq_instrument_identifiers_2
        UNIQUE (instrument_id, identifier_type, identifier_value, valid_from),

    CONSTRAINT fk_instrument_identifiers_instrument_id
        FOREIGN KEY (instrument_id)
        REFERENCES core.instruments(instrument_id),

    CONSTRAINT chk_instrument_identifiers_identifier_type
        CHECK (identifier_type IN ('ISIN','FIGI','CUSIP','SEDOL','TICKER','CIK','LEI_ISSUER')),

    CONSTRAINT fk_instrument_identifiers_exchange_mic
        FOREIGN KEY (exchange_mic)
        REFERENCES ref.exchanges(exchange_mic),

    CONSTRAINT fk_instrument_identifiers_source_id
        FOREIGN KEY (source_id)
        REFERENCES core.data_sources(source_id),

    CONSTRAINT chk_instrument_identifiers_valid_to
        CHECK (valid_to IS NULL OR valid_to > valid_from)
);

CREATE UNIQUE INDEX IF NOT EXISTS uq_instrument_identifiers_3
ON core.instrument_identifiers (instrument_id, identifier_type)
WHERE is_primary = TRUE AND valid_to IS NULL;

CREATE INDEX IF NOT EXISTS idx_instrument_identifiers_1
ON core.instrument_identifiers (identifier_type, identifier_value);
"""

execute_query(query)


✓ Sweet! Data updated


In [16]:
# CORE.instrument_source_map

query = """
CREATE TABLE IF NOT EXISTS core.instrument_source_map (
    instrument_source_map_id      BIGINT GENERATED ALWAYS AS IDENTITY         PRIMARY KEY,
    instrument_id                 BIGINT                                      NOT NULL,
    source_id                     BIGINT                                      NOT NULL,
    source_symbol                 VARCHAR(80)                                 NOT NULL,
    source_exchange_code          VARCHAR(20),
    data_domain                   VARCHAR(30)                                 NOT NULL,
    priority_order                SMALLINT                                    NOT NULL DEFAULT 1,
    valid_from                    DATE                                        NOT NULL,
    valid_to                      DATE,
    is_primary                    BOOLEAN                                     NOT NULL DEFAULT FALSE,
    notes                         TEXT,

    CONSTRAINT uq_instrument_source_map_2
        UNIQUE (instrument_id, source_id, data_domain, source_symbol, valid_from),

    CONSTRAINT fk_instrument_source_map_instrument_id
        FOREIGN KEY (instrument_id)
        REFERENCES core.instruments(instrument_id),

    CONSTRAINT fk_instrument_source_map_source_id
        FOREIGN KEY (source_id)
        REFERENCES core.data_sources(source_id),

    CONSTRAINT chk_instrument_source_map_data_domain
        CHECK (data_domain IN ('PRICE','REFERENCE','CORPORATE_ACTION','NAV','FX')),

    CONSTRAINT chk_instrument_source_map_priority_order
        CHECK (priority_order > 0),

    CONSTRAINT chk_instrument_source_map_valid_to
        CHECK (valid_to IS NULL OR valid_to > valid_from)
);

CREATE UNIQUE INDEX IF NOT EXISTS uq_instrument_source_map_3
ON core.instrument_source_map (instrument_id, data_domain)
WHERE is_primary = TRUE AND valid_to IS NULL;

CREATE INDEX IF NOT EXISTS idx_instrument_source_map_1
ON core.instrument_source_map (source_id, source_symbol);

CREATE INDEX IF NOT EXISTS idx_instrument_source_map_2
ON core.instrument_source_map (instrument_id, data_domain, valid_from, valid_to);
"""

execute_query(query)


✓ Sweet! Data updated


In [17]:
# CORE.instrument_classification

query = """
CREATE TABLE IF NOT EXISTS core.instrument_classification (
    classification_id             BIGINT GENERATED ALWAYS AS IDENTITY         PRIMARY KEY,
    instrument_id                 BIGINT                                      NOT NULL,
    asset_class                   VARCHAR(40)                                 NOT NULL,
    sub_asset_class               VARCHAR(50)                                 NOT NULL,
    sector                        VARCHAR(50),
    industry                      VARCHAR(80),
    geographic_area               VARCHAR(40),
    exposure_country_code         CHAR(2),
    valid_from                    DATE                                        NOT NULL,
    valid_to                      DATE,
    is_current                    BOOLEAN                                     NOT NULL DEFAULT TRUE,
    batch_id                      BIGINT,

    CONSTRAINT uq_instrument_classification_2
        UNIQUE (instrument_id, valid_from),

    CONSTRAINT fk_instrument_classification_instrument_id
        FOREIGN KEY (instrument_id)
        REFERENCES core.instruments(instrument_id),

    CONSTRAINT fk_instrument_classification_exposure_country_code
        FOREIGN KEY (exposure_country_code)
        REFERENCES ref.countries(country_code),

    CONSTRAINT chk_instrument_classification_valid_to
        CHECK (valid_to IS NULL OR valid_to > valid_from),

    CONSTRAINT chk_instrument_classification_is_current
        CHECK (is_current = (valid_to IS NULL)),

    CONSTRAINT fk_instrument_classification_batch_id
        FOREIGN KEY (batch_id)
        REFERENCES audit.etl_batches(batch_id)
);

CREATE UNIQUE INDEX IF NOT EXISTS uq_instrument_classification_3
ON core.instrument_classification (instrument_id)
WHERE is_current = TRUE;

CREATE INDEX IF NOT EXISTS idx_instrument_classification_1
ON core.instrument_classification (instrument_id, valid_from, valid_to);
"""

execute_query(query)


✓ Sweet! Data updated


In [18]:
# CORE.instrument_equity_details

query = """
CREATE TABLE IF NOT EXISTS core.instrument_equity_details (
    instrument_id                 BIGINT                                      PRIMARY KEY,
    share_class                   VARCHAR(20),
    dividend_frequency            VARCHAR(20),
    voting_rights                 BOOLEAN,
    fiscal_year_end_month         SMALLINT,

    CONSTRAINT fk_instrument_equity_details_instrument_id
        FOREIGN KEY (instrument_id)
        REFERENCES core.instruments(instrument_id),

    CONSTRAINT chk_instrument_equity_details_dividend_frequency
        CHECK (dividend_frequency IS NULL OR dividend_frequency IN ('ANNUAL','SEMIANNUAL','QUARTERLY','MONTHLY','IRREGULAR')),

    CONSTRAINT chk_instrument_equity_details_fiscal_year_end_month
        CHECK (fiscal_year_end_month IS NULL OR fiscal_year_end_month BETWEEN 1 AND 12)
);
"""

execute_query(query)


✓ Sweet! Data updated


In [19]:
# CORE.instrument_bond_details

query = """
CREATE TABLE IF NOT EXISTS core.instrument_bond_details (
    instrument_id                 BIGINT                                      PRIMARY KEY,
    bond_type                     VARCHAR(30)                                 NOT NULL,
    coupon_type                   VARCHAR(20)                                 NOT NULL,
    coupon_rate                   NUMERIC(10,6),
    coupon_frequency              SMALLINT,
    face_value                    NUMERIC(20,4)                               NOT NULL,
    day_count_convention          VARCHAR(20),
    seniority                     VARCHAR(30),
    is_callable                   BOOLEAN                                     NOT NULL DEFAULT FALSE,
    is_inflation_linked           BOOLEAN                                     NOT NULL DEFAULT FALSE,

    CONSTRAINT chk_instrument_bond_details_table_3
        CHECK (coupon_type <> 'ZERO_COUPON' OR COALESCE(coupon_frequency,0)=0),

    CONSTRAINT chk_instrument_bond_details_table_4
        CHECK (coupon_type <> 'FIXED' OR coupon_rate IS NOT NULL),

    CONSTRAINT fk_instrument_bond_details_instrument_id
        FOREIGN KEY (instrument_id)
        REFERENCES core.instruments(instrument_id),

    CONSTRAINT chk_instrument_bond_details_bond_type
        CHECK (bond_type IN ('GOVERNMENT','CORPORATE','AGENCY','SUPRANATIONAL')),

    CONSTRAINT chk_instrument_bond_details_coupon_type
        CHECK (coupon_type IN ('FIXED','FLOATING','ZERO_COUPON')),

    CONSTRAINT chk_instrument_bond_details_coupon_rate
        CHECK (coupon_rate IS NULL OR coupon_rate >= 0),

    CONSTRAINT chk_instrument_bond_details_coupon_frequency
        CHECK (coupon_frequency IS NULL OR coupon_frequency IN (1,2,4,12)),

    CONSTRAINT chk_instrument_bond_details_face_value
        CHECK (face_value > 0),

    CONSTRAINT chk_instrument_bond_details_day_count_convention
        CHECK (day_count_convention IS NULL OR day_count_convention IN ('ACT_ACT','30_360','ACT_360','ACT_365'))
);
"""

execute_query(query)


✓ Sweet! Data updated


In [20]:
# CORE.instrument_fund_details

query = """
CREATE TABLE IF NOT EXISTS core.instrument_fund_details (
    instrument_id                 BIGINT                                      PRIMARY KEY,
    fund_type                     VARCHAR(20)                                 NOT NULL,
    distribution_policy           VARCHAR(20),
    replication_method            VARCHAR(20),
    management_style              VARCHAR(20),
    ter                           NUMERIC(8,6),
    is_ucits                      BOOLEAN,
    nav_frequency                 VARCHAR(20),
    reference_benchmark_id        BIGINT,
    fund_domicile                 CHAR(2),

    CONSTRAINT fk_instrument_fund_details_instrument_id
        FOREIGN KEY (instrument_id)
        REFERENCES core.instruments(instrument_id),

    CONSTRAINT chk_instrument_fund_details_fund_type
        CHECK (fund_type IN ('ETF','MUTUAL_FUND')),

    CONSTRAINT chk_instrument_fund_details_distribution_policy
        CHECK (distribution_policy IS NULL OR distribution_policy IN ('ACCUMULATION','DISTRIBUTION')),

    CONSTRAINT chk_instrument_fund_details_replication_method
        CHECK (replication_method IS NULL OR replication_method IN ('PHYSICAL','SYNTHETIC','SAMPLING')),

    CONSTRAINT chk_instrument_fund_details_management_style
        CHECK (management_style IS NULL OR management_style IN ('ACTIVE','PASSIVE')),

    CONSTRAINT chk_instrument_fund_details_ter
        CHECK (ter IS NULL OR (ter >= 0 AND ter <= 1)),

    CONSTRAINT chk_instrument_fund_details_nav_frequency
        CHECK (nav_frequency IS NULL OR nav_frequency IN ('DAILY','WEEKLY','MONTHLY')),

    CONSTRAINT fk_instrument_fund_details_reference_benchmark_id
        FOREIGN KEY (reference_benchmark_id)
        REFERENCES core.benchmarks(benchmark_id),

    CONSTRAINT fk_instrument_fund_details_fund_domicile
        FOREIGN KEY (fund_domicile)
        REFERENCES ref.countries(country_code)
);
"""

execute_query(query)


✓ Sweet! Data updated


In [ ]:
# #